# Stage 2 — Save a reproducible data snapshot

**What:** Collect every request from one fixed historical day and keep an exact copy.

**Why:** NYC updates the source. Training from a saved copy lets us reproduce a result even if the source later changes.

We will save `requests.jsonl` (the rows) and `manifest.json` (how, when, and where we got them). This lesson uses only free Colab, Google Drive, Python, and public data.

In [ ]:
# Cell 2 — Why: keep our files after Colab's temporary runtime ends.
from google.colab import drive
from pathlib import Path
import requests

drive.mount("/content/drive")
folder = Path("/content/drive/MyDrive/nyc311_mlops")
folder.mkdir(parents=True, exist_ok=True)

day_start = "2025-01-15T00:00:00"
day_end = "2025-01-16T00:00:00"
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
where = f"created_date >= '{day_start}' AND created_date < '{day_end}'"
print("Saving under:", folder)

In [ ]:
# Cell 3 — Why: know how many rows the day contains before downloading.
count_reply = requests.get(
    url, params={"$select": "count(*) as n", "$where": where}, timeout=60
)
count_reply.raise_for_status()
expected_rows = int(count_reply.json()[0]["n"])
if not 0 < expected_rows <= 50000:
    raise ValueError("This day needs paging; tell me the row count")
print("Rows expected:", expected_rows)

In [ ]:
# Cell 4 — Why: get the full day and reject an incomplete download.
fields = ["unique_key", "created_date", "closed_date", "complaint_type", "borough"]
params = {
    "$select": ",".join(fields), "$where": where,
    "$order": "created_date ASC, unique_key ASC", "$limit": expected_rows,
}
data_reply = requests.get(url, params=params, timeout=120)
data_reply.raise_for_status()
records = data_reply.json()

ids = [row["unique_key"] for row in records]
if len(records) != expected_rows:
    raise ValueError("The source changed or rows are missing")
if len(ids) != len(set(ids)):
    raise ValueError("Duplicate request IDs found")
print("Downloaded and checked:", len(records), "requests")

In [ ]:
# Cell 5 — Why: keep the exact bytes used in this project.
import hashlib
import json
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_folder = folder / "raw" / run_id
run_folder.mkdir(parents=True, exist_ok=False)
raw_file = run_folder / "requests.jsonl"
raw_bytes = "".join(
    json.dumps(row, sort_keys=True) + "\n" for row in records
).encode("utf-8")
raw_file.write_bytes(raw_bytes)
checksum = hashlib.sha256(raw_bytes).hexdigest()
print("Saved raw data:", raw_file)

In [ ]:
# Cell 6 — Why: document and verify exactly what we saved.
import sys
from importlib.metadata import version

manifest = {
    "source": url, "query": params, "row_count": len(records),
    "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    "sha256": checksum, "lesson_version": "stage2-short-v1",
    "python": sys.version.split()[0], "requests": version("requests"),
}
manifest_file = run_folder / "manifest.json"
manifest_file.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

if hashlib.sha256(raw_file.read_bytes()).hexdigest() != manifest["sha256"]:
    raise ValueError("Saved file does not match its checksum")
print("Rows saved:", manifest["row_count"])
print("Checksum verified: True")
print("Manifest:", manifest_file)

# Stage 2 takeaway and quiz

**Why this stage mattered:** A model can only be reproduced when we know the exact data it learned from. The saved file is our snapshot; the manifest records its origin and fingerprint. Later we will also pin library versions and record the code commit used for training.

**Easy quiz — answer in your own words:**

1. Why did we choose a fixed day instead of the newest ten requests?
2. Why do we check that the downloaded row count matches the expected count?
3. If the source changes next month, which file lets us reuse today's exact data?
4. What would a changed checksum tell us?

Send me the outputs of Cells 3, 4, and 6 and your answers. We will then build the 72-hour label in Stage 3.